# Classification Preprocessing
This notebook loads the cleaned dataset and applies classification-specific preprocessing steps.

In [8]:
!pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.4 MB 6.0 MB/s eta 0:00:02
   ------------- -------------------------- 2.9/8.4 MB 7.4 MB/s eta 0:00:01
   ----------------------- ---------------- 5.0/8.4 MB 8.1 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.4 MB 7.5 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.4 MB 6.4 MB/s eta 0:00:01
   --------------------------------- ------ 7.1/8.4 MB 5.5 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.4 MB 5.4 MB/s eta 0:00:01
   -------------------------------------- - 8.1/8.4 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 4.6 MB/s  0:00:01
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
    --------------------------------------- 0.5/37.4 MB 3.7 MB/s eta 0:00:10
   - -----------------------------

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os

import warnings
warnings.filterwarnings('ignore')

## Load the Cleaned Dataset

In [10]:
DATA_PATH = "../data/RT_IOT2022_cleaned.csv"
df = pd.read_csv(DATA_PATH)
print("Data shape after loading:", df.shape)

Data shape after loading: (117922, 83)


## 1. Feature Engineering: `avg_payload_per_packet`
**Justification:** Helps the model distinguish between empty-packet attacks (like SYN floods) and data-heavy traffic.

In [11]:
total_pkts = df['fwd_pkts_tot'] + df['bwd_pkts_tot']
# Avoid division by zero
df['avg_payload_per_packet'] = np.where(total_pkts > 0, df['flow_pkts_payload.tot'] / total_pkts, 0)

print("Engineered feature 'avg_payload_per_packet' added.")

Engineered feature 'avg_payload_per_packet' added.


## 2. Outlier Treatment: Percentile Capping
We cap numerical outliers at the 1st and 99th percentiles instead of dropping them to prevent losing minority class samples.

In [12]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower, upper)
    
print("Outliers capped at 1st and 99th percentiles.")

Outliers capped at 1st and 99th percentiles.


## 3. Encoding
- **Label Encode** the target variable (`Attack_type`).
- **One-Hot Encode** nominal categorical variables.

In [13]:
# Label encode the target variable
le = LabelEncoder()
df['Attack_type_encoded'] = le.fit_transform(df['Attack_type'])

# Store the mapping for future reference
target_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Target mapping:", target_mapping)

# Drop original target column and isolate X and y
y = df['Attack_type_encoded']
X = df.drop(columns=['Attack_type', 'Attack_type_encoded'])

# One-hot encode nominal categorical variables ('proto', 'service' if they are present as objects)
categorical_cols = X.select_dtypes(include=['object']).columns
print("Categorical columns to one-hot encode:", list(categorical_cols))

if len(categorical_cols) > 0:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print("Shape of X after encoding:", X.shape)

Target mapping: {'ARP_poisioning': np.int64(0), 'DDOS_Slowloris': np.int64(1), 'DOS_SYN_Hping': np.int64(2), 'MQTT_Publish': np.int64(3), 'Metasploit_Brute_Force_SSH': np.int64(4), 'NMAP_FIN_SCAN': np.int64(5), 'NMAP_OS_DETECTION': np.int64(6), 'NMAP_TCP_scan': np.int64(7), 'NMAP_UDP_SCAN': np.int64(8), 'NMAP_XMAS_TREE_SCAN': np.int64(9), 'Thing_Speak': np.int64(10), 'Wipro_bulb': np.int64(11)}
Categorical columns to one-hot encode: ['proto', 'service']
Shape of X after encoding: (117922, 92)


## 4. Stratified Train-Test Split (80:20)
Stratification ensures a fair representation of all target classes in both the train and test sets.

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (94337, 92)
X_test shape: (23585, 92)


## 5. Scaling
We fit the `StandardScaler` on the **training set only**, and then use it to transform both the train and test sets. This prevents data leakage (as per section 7.1 of guidelines).

In [15]:
scaler = StandardScaler()

# Get numeric columns (everything in X is numeric now)
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Scaling completed.")

Scaling completed.


## 6. Save Preprocessed Data
Save the final datasets to the `data/classification/` folder for use in model training.

In [16]:
import os
os.makedirs("../data/classification", exist_ok=True)

X_train_scaled.to_csv("../data/classification/X_train.csv", index=False)
X_test_scaled.to_csv("../data/classification/X_test.csv", index=False)
y_train.to_csv("../data/classification/y_train.csv", index=False)
y_test.to_csv("../data/classification/y_test.csv", index=False)

print("Preprocessed classification datasets saved to ../data/classification/")

Preprocessed classification datasets saved to ../data/classification/
